# Donut Base Model Test

`naver-clova-ix/donut-base-finetuned-cord-v2` 모델을 사용하여 CORD-v2(영수증 파싱) 태스크를 테스트합니다.

In [2]:
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Python: 3.11.15 (main, Mar 11 2026, 17:11:19) [GCC 14.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GB10


In [3]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

from datasets import load_dataset
from PIL import Image
import torch, json, re
import matplotlib.pyplot as plt
import matplotlib as mpl

## Base Model vs Fine-tuned Model

### Base Model (`donut-base`)
대량의 문서 이미지로 **일반적인 패턴**을 학습한 모델

| 항목 | 내용 |
|------|------|
| 학습 데이터 | IIT-CDIP (1100만 장의 다양한 문서 이미지) |
| 학습 목표 | 이미지에서 텍스트를 읽는 능력 자체를 익힘 |
| 결과 | 문서를 "보고 이해"할 수 있지만 특정 태스크는 모름 |

### Fine-tuned Model (`donut-base-finetuned-cord-v2`)
Base Model을 출발점으로, **특정 태스크 데이터**로 추가 학습한 모델

| 항목 | 내용 |
|------|------|
| 학습 데이터 | CORD-v2 (영수증 800장 + 정답 JSON) |
| 학습 목표 | 영수증 → `gt_parse` JSON 구조로 변환 |
| 결과 | 영수증 파싱에 특화된 능력 보유 |

### 비유

```
Base Model   = 대학 졸업생  (읽기·이해력은 갖춤, 전문 업무는 아직 모름)
Fine-tuning  = 입사 후 실무 교육  (영수증 처리 업무만 집중 훈련)
Fine-tuned   = 영수증 처리 전문 직원  (영수증 → JSON 즉시 변환 가능)
```

### 학습 흐름

```
[Base Model 사전학습]
대용량 문서 → 일반 표현 학습
        ↓
[Fine-tuning]
CORD-v2 영수증 800장 + 정답 JSON → 태스크 특화 학습
        ↓
[Fine-tuned Model]
영수증 이미지 + <s_cord-v2> 태스크 토큰 → JSON 출력
```

In [ ]:
model_name = "naver-clova-ix/donut-base-finetuned-cord-v2"

# DonutProcessor: 이미지 전처리(resize, normalize) + 토크나이저를 하나로 묶은 클래스
processor = DonutProcessor.from_pretrained(model_name)

# VisionEncoderDecoderModel: Swin Transformer(encoder) + mBART(decoder) 구조
model = VisionEncoderDecoderModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()  # 추론 모드: dropout 비활성화, gradient 계산 불필요
print(f"Model loaded on {device}")

## CORD-v2 Dataset 구조

영수증 이미지 + JSON 어노테이션으로 구성된 문서 파싱 벤치마크 데이터셋

### Splits

| Split      | 샘플 수 |
|------------|--------|
| train      | 800    |
| validation | 100    |
| test       | 100    |

### 컬럼

| 컬럼명          | 타입        | 설명                      |
|----------------|-------------|---------------------------|
| `image`        | PIL Image   | 영수증 이미지              |
| `ground_truth` | JSON string | 레이블 + 메타 정보 전체    |

### `ground_truth` JSON 구조

```
ground_truth
├── gt_parse              ← Donut 학습에 사용하는 핵심 레이블
│   ├── menu []           ← 주문 항목 목록
│   │   ├── nm                메뉴명
│   │   ├── cnt               수량
│   │   ├── price             가격
│   │   ├── unitprice         단가
│   │   ├── itemsubtotal      항목 소계
│   │   ├── num               번호
│   │   ├── discountprice     할인가
│   │   ├── vatyn             부가세 여부
│   │   ├── sub               하위 항목
│   │   └── etc               기타
│   ├── sub_total         ← 소계 영역
│   │   ├── subtotal_price    소계
│   │   ├── discount_price    할인
│   │   ├── service_price     서비스료
│   │   ├── tax_price         세금
│   │   ├── othersvc_price    기타 서비스료
│   │   └── etc               기타
│   ├── total             ← 합계 영역
│   │   ├── total_price       총액
│   │   ├── creditcardprice   카드 결제
│   │   ├── cashprice         현금 결제
│   │   ├── changeprice       거스름돈
│   │   ├── emoneyprice       전자화폐
│   │   ├── menutype_cnt      메뉴 종류 수
│   │   ├── menuqty_cnt       총 수량
│   │   └── total_etc         기타
│   └── void_menu []      ← 취소 항목 (선택적)
├── meta                  ← image_id, image_size, split 등 메타 정보
├── valid_line []         ← 단어별 바운딩 박스 좌표 (OCR 전용, Donut 미사용)
├── roi                   ← 관심 영역 좌표
├── repeating_symbol      ← 반복 기호
└── dontcare []           ← 무시할 영역
```

> Donut은 `gt_parse` 만 사용하며, `valid_line` / `roi` 등은 다른 OCR 모델용 어노테이션입니다.

In [ ]:
# CORD-v2: 영수증 이미지 + JSON 어노테이션 데이터셋 (donut-base의 학습 도메인)
# test[:1] → 테스트셋 첫 번째 샘플 1장만 로드
dataset = load_dataset("naver-clova-ix/cord-v2", split="test[:1]")
print(dataset)                           # Dataset은 .shape 없음 → print로 확인
print(f"샘플 수: {len(dataset)}, 컬럼 수: {len(dataset.features)}")

image = dataset[0]["image"]
ground_truth = json.loads(dataset[0]["ground_truth"])

# 샘플 구조 출력
print("\n=== 이미지 정보 ===")
print(f"크기: {image.size}  모드: {image.mode}")

print("\n=== ground_truth 최상위 키 ===")
print(list(ground_truth.keys()))

print("\n=== gt_parse 키 ===")
gt = ground_truth["gt_parse"]
print(list(gt.keys()))

print("\n=== gt_parse 내용 ===")
print(json.dumps(gt, indent=2, ensure_ascii=False))

plt.figure(figsize=(6, 8))
plt.imshow(image)
plt.axis("off")
plt.title("Test Image (CORD-v2)")
plt.show()

### 로컬 캐시 위치

`load_dataset` 최초 실행 시 HuggingFace Hub에서 자동 다운로드되며, 이후 호출은 캐시에서 즉시 로드됩니다.

```
~/.cache/huggingface/datasets/naver-clova-ix___cord-v2/
└── default/0.0.0/7f0115a4.../
    ├── cord-v2-train-00000-of-00002.arrow   ← train (절반)
    ├── cord-v2-train-00001-of-00002.arrow   ← train (절반)
    ├── cord-v2-validation.arrow             ← validation
    ├── cord-v2-test.arrow                   ← test
    └── dataset_info.json                    ← 메타 정보
```

| 항목   | 내용                                        |
|--------|---------------------------------------------|
| 총 용량 | 2.2 GB                                     |
| 포맷   | `.arrow` (Apache Arrow — 빠른 컬럼형 포맷)  |

In [12]:
# task_prompt: 모델에게 수행할 태스크를 지정하는 특수 토큰
# <s_cord-v2>는 영수증 파싱 태스크를 의미하며, 디코더의 첫 번째 입력으로 사용됨
task_prompt = "<s_cord-v2>"
decoder_input_ids = processor.tokenizer(
    task_prompt, add_special_tokens=False, return_tensors="pt"
).input_ids.to(device)

# processor: 이미지를 모델 입력 크기(1280×960)로 리사이즈 후 [-1, 1] 범위로 정규화
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

with torch.no_grad():  # 추론 시 gradient 계산 생략 → 메모리 절약
    outputs = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=model.decoder.config.max_position_embeddings,  # 최대 시퀀스 길이
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,            # EOS 토큰 도달 시 생성 중단
        use_cache=True,                                            # KV-cache로 디코딩 속도 향상
        bad_words_ids=[[processor.tokenizer.unk_token_id]],       # UNK 토큰 생성 금지
        return_dict_in_generate=True,
    )

# 생성된 토큰 ID → 문자열로 디코딩
sequence = processor.batch_decode(outputs.sequences)[0]
# 특수 토큰(EOS, PAD) 제거 
sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(
    processor.tokenizer.pad_token, ""
)
# 첫 번째 태스크 토큰(<s_cord-v2> 등)을 정규식으로 찾아서 제거합니다.
# 예: "<s_cord-v2>...실제내용..." → "...실제내용..."
sequence = re.sub(r"<.*?>", "", sequence, count=1).strip()
print("Raw output:", sequence)

Raw output: - 0. 0. 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

In [ ]:
def token2json(tokens, is_inner_value=False):
    """XML-like 토큰 시퀀스를 Python dict로 변환
    예: <s_total>12500</s_total> → {"total": "12500"}
    중첩 구조(dict 안의 dict)도 재귀적으로 처리
    """
    output = {}
    while tokens:
        start_token = re.search(r"<s_(.*?)>", tokens)
        if not start_token:
            break
        key = start_token.group(1)
        end_token = re.search(rf"</s_{key}>", tokens)
        value = tokens[start_token.end(): end_token.start() if end_token else len(tokens)]
        value = value.strip()
        if re.search(r"<s_", value):          # 중첩 태그가 있으면 재귀 파싱
            value = token2json(value, is_inner_value=True)
            if value:
                output[key] = value
        else:
            output[key] = value               # 리프 노드: 문자열 값 저장
        tokens = tokens[end_token.end():].strip() if end_token else ""
    return output

result = token2json(sequence)
print("Parsed result:")
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# 데이터셋에 포함된 정답 레이블과 모델 예측 결과를 비교
print("Ground Truth:")
print(json.dumps(ground_truth, indent=2, ensure_ascii=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

axes[0].imshow(image)
axes[0].axis("off")
axes[0].set_title("Input Image")

axes[1].axis("off")
axes[1].set_title("Parsed Output")
axes[1].text(
    0.05, 0.95,
    json.dumps(result, indent=2, ensure_ascii=False),
    transform=axes[1].transAxes,
    fontsize=8,
    verticalalignment="top",
    fontfamily="monospace"
)

plt.tight_layout()
plt.show()